## 05 Water Availability: Groundwater Monitoring Coverage and Trends
**Series:** Tribal Agriculture & Land Health          
**Author:** Lilly Jones, PhD  
**Primary Focus:** Oglala Lakota (Pine Ridge), Sicangu Lakota (Rosebud)  
**In Scope:** All South Dakota Tribal Nations  
**Data Sources:** USGS NWIS (groundwater sites and water levels), Census TIGER AIANNH

## Purpose
Water is the binding constraint on agriculture in the mixed-grass prairie of
southern South Dakota. Whether livestock can graze a pasture, how quickly
vegetation recovers from drought, and whether a bison herd can be sustained
on a given land base all depend on reliable water access. Groundwater is the 
primary water source for livestock on most Pine Ridge and Rosebud pastures.

This notebook uses the USGS National Water Information System (NWIS) to:
- Map the distribution of USGS groundwater monitoring wells on and near SD Tribal lands
- Quantify monitoring coverage gaps
- Extract water level time series where available
- Identify long-term trends in groundwater levels

## Monitoring Gap as Equity Finding
> USGS groundwater monitoring well coverage is systematically sparse on South
> Dakota Tribal lands relative to surrounding non-Tribal areas. This is not
> evidence of less groundwater, it is evidence of less federal investment in
> monitoring infrastructure on Tribal lands. The monitoring gap itself is a
> finding with policy implications.

Where USGS monitoring is absent, Tribal water programs rely on their own
well-level observations, the data collected in the operational pipeline
track of this repository (`data/raw/groundwater.csv`).

## Research Questions
- How many active USGS groundwater monitoring sites exist on or within
  50 km of each SD Tribal Nation?
- Where are the monitoring dead zones on Tribal lands?
- For wells with long-term records, is groundwater declining? And does
  the decline correlate with the drought years identified in notebook 02?

In [ ]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
from datetime import datetime

import contextily as ctx
import geopandas as gpd
gpd.options.io_engine = "fiona"
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from scipy import stats
from shapely.geometry import Point
from shapely.validation import make_valid
from tenacity import retry, stop_after_attempt, wait_exponential

from src.data import constants
from src.data.constants import (
    SD_TRIBES_ALL, SD_TRIBES_PRIMARY,
    CENSUS_NAME_MAP, CENSUS_TO_COMMON,
    CRS_GEOGRAPHIC, CRS_PROJECTED,
)
from src.indigenous.sovereignty import print_data_acknowledgment, generate_citations

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

_retry = retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    reraise=True,
)

print(f"Repo root : {REPO_ROOT}")
print(f"Analysis run: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

In [ ]:
# Print data acknowledgement at the top of every notebook
print_data_acknowledgment(source_keys=["census_aiannh", "usgs_nwis_groundwater"])

## Configure

In [ ]:
# Analysis parameters 
# Buffer around Tribal land to search for nearby monitoring wells (km)
SEARCH_BUFFER_KM = 50

# Coverage gap thresholds
ADEQUATE_COVERAGE_KM = 20   # nearest well ≤ 20 km = adequate
MARGINAL_COVERAGE_KM = 50   # nearest well 20–50 km = marginal
# > 50 km = monitoring gap

# Minimum record length for trend analysis (years)
MIN_RECORD_YEARS = 10

# USGS NWIS REST API
NWIS_SITE_URL   = f"{constants.USGS_NWIS_BASE}/site/"
NWIS_GWL_URL    = f"{constants.USGS_NWIS_BASE}/gwlevels/"
NWIS_DV_URL     = f"{constants.USGS_NWIS_BASE}/dv/"

print("WATER AVAILABILITY CONFIGURATION")
print(f"  Search buffer  : {SEARCH_BUFFER_KM} km around each Tribal land")
print(f"  Adequate       : nearest well ≤ {ADEQUATE_COVERAGE_KM} km")
print(f"  Marginal       : nearest well {ADEQUATE_COVERAGE_KM}–{MARGINAL_COVERAGE_KM} km")
print(f"  Gap            : nearest well > {MARGINAL_COVERAGE_KM} km")
print(f"  Min record     : {MIN_RECORD_YEARS} years for trend analysis")

## Load Tribal Boundaries

In [ ]:
# Tribal boundaries
GEOJSON_PATH = constants.OUTPUTS_DIR/"sd_tribal_land_base.geojson"
CACHE_PATH   = constants.CACHE_DIR/"tl_2023_us_aiannh.geojson"

if GEOJSON_PATH.exists():
    tribal_lands = gpd.read_file(GEOJSON_PATH)
    print(f"Loaded from notebook 01 output: {len(tribal_lands)} Tribal Nations")
elif CACHE_PATH.exists():
    all_aiannh   = gpd.read_file(CACHE_PATH)
    census_names = list(CENSUS_NAME_MAP.values())
    tribal_lands = all_aiannh[all_aiannh["NAME"].isin(census_names)].copy()
    tribal_lands = tribal_lands.dissolve(by="NAME", as_index=False)
    tribal_lands["geometry"]    = tribal_lands.geometry.apply(make_valid)
    tribal_lands["common_name"] = tribal_lands["NAME"].map(CENSUS_TO_COMMON)
    tribal_lands["area_km2"]    = tribal_lands.to_crs(CRS_PROJECTED).geometry.area / 1e6
    tribal_lands["is_primary"]  = tribal_lands["common_name"].isin(SD_TRIBES_PRIMARY)
else:
    raise FileNotFoundError("Run notebook 01 first to create sd_tribal_land_base.geojson")

# 50 km search buffer per Tribal land (projected to geographic)
tribal_proj    = tribal_lands.to_crs(CRS_PROJECTED)
buffer_proj    = tribal_proj.copy()
buffer_proj["geometry"] = tribal_proj.geometry.buffer(SEARCH_BUFFER_KM * 1000)
buffer_geo     = buffer_proj.to_crs(CRS_GEOGRAPHIC)

# Study area bbox (union of all buffered areas)
study_bounds   = buffer_geo.total_bounds
STUDY_BBOX_STR = f"{study_bounds[0]:.4f},{study_bounds[1]:.4f},{study_bounds[2]:.4f},{study_bounds[3]:.4f}"

print(f"Study area bbox: {STUDY_BBOX_STR}")

## Fetch USGS NWIS Groundwater Monitoring Sites

In [ ]:
# USGS NWIS site inventory
# Uses the NWIS site service (RDB format: plain text, no JSON parsing issues)
# siteType=GW  = groundwater wells
# hasDataTypeCd=gw = has groundwater level data

SITES_CACHE = constants.CACHE_DIR/"nwis_gw_sites_sd.csv"

try:
    constants.CACHE_DIR.mkdir(parents=True, exist_ok=True)
except FileExistsError:
    pass

@_retry
def fetch_nwis_sites(bbox_str: str) -> gpd.GeoDataFrame:
    """Fetch USGS groundwater monitoring sites by bounding box (RDB format)."""
    r = requests.get(
        NWIS_SITE_URL,
        params={
            "format":         "rdb",
            "bBox":           bbox_str,
            "siteType":       "GW",
            "hasDataTypeCd":  "gw",
            "siteStatus":     "all",
        },
        timeout=60,
    )
    r.raise_for_status()

    # RDB format: lines starting with # are comments; two header rows
    lines = [l for l in r.text.splitlines() if not l.startswith("#")]
    if len(lines) < 3:
        return gpd.GeoDataFrame()

    # First non-comment line = column names; second = data types (skip)
    from io import StringIO
    cols = lines[0].split("\t")
    data_lines = [l for l in lines[2:] if l.strip()]
    df = pd.read_csv(
        StringIO("\n".join(data_lines)),
        sep="\t", header=None, names=cols,
        dtype=str, low_memory=False,
    )

    df["dec_lat_va"]  = pd.to_numeric(df.get("dec_lat_va"),  errors="coerce")
    df["dec_long_va"] = pd.to_numeric(df.get("dec_long_va"), errors="coerce")
    df = df.dropna(subset=["dec_lat_va", "dec_long_va"]).reset_index(drop=True)

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["dec_long_va"], df["dec_lat_va"]),
        crs=CRS_GEOGRAPHIC,
    )
    return gdf


if SITES_CACHE.exists():
    sites_raw = pd.read_csv(SITES_CACHE, dtype=str)
    sites_raw["dec_lat_va"]  = pd.to_numeric(sites_raw["dec_lat_va"],  errors="coerce")
    sites_raw["dec_long_va"] = pd.to_numeric(sites_raw["dec_long_va"], errors="coerce")
    sites_gdf = gpd.GeoDataFrame(
        sites_raw,
        geometry=gpd.points_from_xy(sites_raw["dec_long_va"], sites_raw["dec_lat_va"]),
        crs=CRS_GEOGRAPHIC,
    )
    print(f"Loaded {len(sites_gdf):,} NWIS sites from cache")
else:
    print("Fetching USGS NWIS groundwater sites...")
    sites_gdf = fetch_nwis_sites(STUDY_BBOX_STR)
    sites_gdf.drop(columns=["geometry"]).to_csv(SITES_CACHE, index=False)
    print(f"Downloaded and cached {len(sites_gdf):,} sites")

print(f"\nSite type breakdown:")
if "site_tp_cd" in sites_gdf.columns:
    print(sites_gdf["site_tp_cd"].value_counts().to_string())

## Coverage Analysis

In [ ]:
# Distance to nearest USGS well per Tribal land 
tribal_proj  = tribal_lands.to_crs(CRS_PROJECTED)
sites_proj   = sites_gdf.to_crs(CRS_PROJECTED)

coverage_records = []

for _, tribe in tribal_proj.iterrows():
    centroid  = tribe.geometry.centroid
    dists_km  = sites_proj.geometry.distance(centroid) / 1000

    nearest_km  = round(dists_km.min(), 1)
    nearest_idx = dists_km.idxmin()
    nearest_id  = sites_gdf.loc[nearest_idx, "site_no"] if "site_no" in sites_gdf.columns else "unknown"

    count_50km = int((dists_km <= SEARCH_BUFFER_KM).sum())
    count_20km = int((dists_km <= ADEQUATE_COVERAGE_KM).sum())

    # Wells within Tribal boundary
    within = sites_proj[sites_proj.within(tribe.geometry)]

    if nearest_km <= ADEQUATE_COVERAGE_KM:
        coverage_cat = "Adequate"
    elif nearest_km <= MARGINAL_COVERAGE_KM:
        coverage_cat = "Marginal"
    else:
        coverage_cat = "Monitoring Gap"

    coverage_records.append({
        "common_name":      tribe["common_name"],
        "is_primary":       tribe["is_primary"],
        "nearest_well_km":  nearest_km,
        "nearest_site_no":  nearest_id,
        "wells_within":     len(within),
        "wells_within_20km": count_20km,
        "wells_within_50km": count_50km,
        "coverage_category": coverage_cat,
    })

coverage_df = pd.DataFrame(coverage_records)

print("USGS GROUNDWATER MONITORING COVERAGE BY TRIBAL NATION")
print(
    coverage_df[
        ["common_name", "nearest_well_km", "wells_within",
         "wells_within_50km", "coverage_category"]
    ]
    .sort_values("nearest_well_km", ascending=False)
    .to_string(index=False)
)
print(f"\nTribal Nations in monitoring gap (> {MARGINAL_COVERAGE_KM} km to nearest well):")
gaps = coverage_df[coverage_df["coverage_category"] == "Monitoring Gap"]
for _, row in gaps.iterrows():
    print(f"  {row['common_name']}: {row['nearest_well_km']} km to nearest USGS well")

## Water Level Time Series
For wells with long-term records near Pine Ridge and Rosebud, extract
water level data and analyze trends.

In [ ]:
# Identify wells with long records near primary Tribes
# Filter to wells within 50 km of Pine Ridge or Rosebud centroids

primary_proj = tribal_proj[tribal_proj["is_primary"]]

candidate_sites = []
for _, tribe in primary_proj.iterrows():
    centroid = tribe.geometry.centroid
    dists    = sites_proj.geometry.distance(centroid) / 1000
    nearby   = sites_gdf[(dists <= SEARCH_BUFFER_KM).values].copy()
    nearby["tribe_name"] = tribe["common_name"]
    nearby["dist_km"]    = dists[dists <= SEARCH_BUFFER_KM].round(1).values
    candidate_sites.append(nearby)

if candidate_sites:
    candidates = pd.concat(candidate_sites, ignore_index=True).drop_duplicates(subset="site_no")
    print(f"Candidate wells near Pine Ridge/Rosebud: {len(candidates)}")
    # Prefer wells with longer records: check begin_date if available
    if "begin_date" in candidates.columns:
        candidates["begin_date"] = pd.to_datetime(candidates["begin_date"], errors="coerce")
        candidates["record_years"] = (
            (pd.Timestamp.now() - candidates["begin_date"]).dt.days / 365
        ).round(1)
        long_record = candidates[candidates["record_years"] >= MIN_RECORD_YEARS].copy()
        print(f"  With ≥ {MIN_RECORD_YEARS} years of record: {len(long_record)}")
    else:
        long_record = candidates.head(10)  # fallback: take first 10
        print("  begin_date not available, using first 10 candidates")
else:
    long_record = pd.DataFrame()
    print("No candidate wells found within search buffer.")

In [ ]:
# Fetch water level time series
# Uses USGS NWIS groundwater levels service (discrete measurements)
# Limit to 5 wells to avoid long download times

@_retry
def fetch_gwl_timeseries(site_no: str) -> pd.DataFrame:
    """
    Fetch discrete groundwater level measurements for one USGS well.
    Returns DataFrame with date and water_level_ft columns.
    """
    r = requests.get(
        NWIS_GWL_URL,
        params={
            "format":  "rdb",
            "sites":   site_no,
            "startDT": "1980-01-01",
        },
        timeout=60,
    )
    r.raise_for_status()

    from io import StringIO
    lines = [l for l in r.text.splitlines() if not l.startswith("#")]
    if len(lines) < 3:
        return pd.DataFrame()

    cols       = lines[0].split("\t")
    data_lines = [l for l in lines[2:] if l.strip()]
    if not data_lines:
        return pd.DataFrame()

    df = pd.read_csv(
        StringIO("\n".join(data_lines)),
        sep="\t", header=None, names=cols,
        dtype=str, low_memory=False,
    )

    # Date column varies: try common names
    date_col = next(
        (c for c in cols if "lev_dt" in c or "date" in c.lower()), None
    )
    level_col = next(
        (c for c in cols
         if "lev_va" in c or ("depth" in c.lower() and "ft" in c.lower())), None
    )
    if date_col is None or level_col is None:
        return pd.DataFrame()

    df["date"]           = pd.to_datetime(df[date_col], errors="coerce")
    df["water_level_ft"] = pd.to_numeric(df[level_col], errors="coerce")
    df["site_no"]        = site_no

    return (
        df[["site_no", "date", "water_level_ft"]]
        .dropna(subset=["date", "water_level_ft"])
        .reset_index(drop=True)
    )


MAX_WELLS_TO_FETCH = 5
gwl_parts  = []
gwl_failed = []

wells_to_fetch = long_record.head(MAX_WELLS_TO_FETCH) if not long_record.empty else pd.DataFrame()

for _, site in wells_to_fetch.iterrows():
    site_no    = site["site_no"]
    site_name  = site.get("station_nm", site_no)
    cache_file = constants.CACHE_DIR / f"gwl_{site_no}.csv"

    if cache_file.exists():
        df = pd.read_csv(cache_file, parse_dates=["date"])
        print(f"  {site_name}: loaded from cache ({len(df)} measurements)")
    else:
        try:
            df = fetch_gwl_timeseries(site_no)
            if df.empty:
                print(f"  {site_name}: no data")
                continue
            df.to_csv(cache_file, index=False)
            print(f"  {site_name}: {len(df)} measurements ({df['date'].min().year}–{df['date'].max().year})")
        except Exception as e:
            gwl_failed.append(site_no)
            print(f"  {site_name}: failed — {e}")
            continue

    df["site_name"] = site_name
    gwl_parts.append(df)

if gwl_parts:
    gwl_df = pd.concat(gwl_parts, ignore_index=True)
    gwl_df["year"]  = gwl_df["date"].dt.year
    gwl_df["month"] = gwl_df["date"].dt.month
    print(f"\nWater level data: {len(gwl_df):,} measurements across {gwl_df['site_no'].nunique()} wells")
else:
    gwl_df = pd.DataFrame()
    print("\nNo water level data loaded. Coverage analysis only.")

In [ ]:
# Groundwater level trends
# Theil-Sen slope on annual median water depth.
# In USGS data, larger depth-to-water = lower water table.

gwl_trends = []

if not gwl_df.empty:
    for site_no, grp in gwl_df.groupby("site_no"):
        annual = grp.groupby("year")["water_level_ft"].median().reset_index()
        annual = annual.dropna()
        if len(annual) < MIN_RECORD_YEARS:
            continue
        slope, _, _, _ = stats.theilslopes(
            annual["water_level_ft"], annual["year"]
        )
        _, _, _, p, _ = stats.linregress(
            annual["year"], annual["water_level_ft"]
        )
        gwl_trends.append({
            "site_no":          site_no,
            "site_name":        grp["site_name"].iloc[0],
            "n_years":          len(annual),
            "slope_ft_yr":      round(slope, 3),
            "slope_ft_decade":  round(slope * 10, 2),
            "p_value":          round(p, 3),
            "significant":      p < 0.05,
            # Positive slope = depth to water increasing = water table falling
            "direction":        "Declining" if slope > 0 else "Rising / Stable",
        })

if gwl_trends:
    gwl_trend_df = pd.DataFrame(gwl_trends)
    print("GROUNDWATER LEVEL TRENDS (Theil-Sen slope)")
    print("Positive slope = depth to water INCREASING = water table FALLING")
    print(gwl_trend_df[
        ["site_name", "n_years", "slope_ft_decade", "direction", "p_value", "significant"]
    ].to_string(index=False))
else:
    gwl_trend_df = pd.DataFrame()
    print("No wells with sufficient record length for trend analysis.")

## Visualizations

In [ ]:
# USGS wells and Tribal boundaries
COVERAGE_COLORS = {
    "Adequate":       "#27AE60",
    "Marginal":       "#E67E22",
    "Monitoring Gap": "#C0392B",
}

fig, ax = plt.subplots(figsize=(13, 8))

# Tribal lands colored by coverage
tribal_with_cov = tribal_lands.merge(
    coverage_df[["common_name", "coverage_category", "nearest_well_km", "wells_within"]],
    on="common_name", how="left",
)
for cat, color in COVERAGE_COLORS.items():
    sub = tribal_with_cov[tribal_with_cov["coverage_category"] == cat]
    if not sub.empty:
        sub.to_crs(3857).plot(
            ax=ax, color=color, alpha=0.4,
            edgecolor="white", linewidth=1.5,
        )

# USGS wells
sites_in_area = sites_gdf[
    sites_gdf.geometry.within(
        buffer_geo.union_all()
    )
]
if not sites_in_area.empty:
    sites_in_area.to_crs(3857).plot(
        ax=ax, color="#1A5276", marker="^",
        markersize=20, alpha=0.6, edgecolor="white", linewidth=0.3,
        label="USGS monitoring well",
    )

# Labels
for _, row in tribal_with_cov.iterrows():
    cx3, cy3 = (
        gpd.GeoDataFrame(
            geometry=[row.geometry.centroid], crs=CRS_GEOGRAPHIC
        ).to_crs(3857).geometry.iloc[0].coords[0]
    )
    label = (
        f"{row['common_name'].split()[0]}\n"
        f"{row.get('nearest_well_km', '?')} km | {row.get('wells_within', 0)} on-res"
    )
    ax.annotate(
        label, (cx3, cy3), ha="center", fontsize=6.5,
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7),
    )

try:
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.4)
except Exception:
    pass

ax.set_axis_off()
ax.legend(
    handles=[
        mpatches.Patch(color=v, alpha=0.5, label=k)
        for k, v in COVERAGE_COLORS.items()
    ] + [
        plt.Line2D([0],[0], marker="^", color="w",
                   markerfacecolor="#1A5276", markersize=8,
                   label="USGS monitoring well"),
    ],
    loc="lower left", fontsize=8,
)
ax.set_title(
    "USGS Groundwater Monitoring Coverage for South Dakota Tribal Lands\n"
    "Color = coverage category | Labels: nearest well km | wells on-reservation",
    fontsize=11, fontweight="bold",
)
plt.tight_layout()
try:
    fig_dir = constants.OUTPUTS_DIR / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(fig_dir/"05_groundwater_coverage_map.png", dpi=150, bbox_inches="tight")
except Exception:
    pass
plt.show()

In [ ]:
# Coverage gap bar chart
fig, ax = plt.subplots(figsize=(10, 6))

s = coverage_df.sort_values("nearest_well_km", ascending=False)
bar_colors = [COVERAGE_COLORS.get(c, "gray") for c in s["coverage_category"]]

ax.barh(s["common_name"], s["nearest_well_km"], color=bar_colors, alpha=0.85)
ax.axvline(ADEQUATE_COVERAGE_KM, color="#27AE60", linestyle="--",
           alpha=0.7, linewidth=1.2, label=f"Adequate (≤ {ADEQUATE_COVERAGE_KM} km)")
ax.axvline(MARGINAL_COVERAGE_KM, color="#E67E22", linestyle="--",
           alpha=0.7, linewidth=1.2, label=f"Marginal (≤ {MARGINAL_COVERAGE_KM} km)")

ax.set_xlabel("Distance to nearest USGS groundwater monitoring well (km)", fontsize=10)
ax.set_title(
    "Groundwater Monitoring Coverage Gaps for South Dakota Tribal Nations",
    fontsize=11, fontweight="bold",
)
ax.legend(fontsize=8)
sns.despine(ax=ax)
plt.tight_layout()
try:
    fig.savefig(fig_dir/"05_groundwater_coverage_gaps.png", dpi=150, bbox_inches="tight")
except Exception:
    pass
plt.show()

In [ ]:
# Water level time series
if not gwl_df.empty:
    sites_in_data = gwl_df["site_no"].unique()
    n     = len(sites_in_data)
    ncols = min(2, n)
    nrows = (n + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(13, nrows * 4))
    axes = np.array(axes).flatten()

    for i, site_no in enumerate(sites_in_data):
        ax   = axes[i]
        grp  = gwl_df[gwl_df["site_no"] == site_no].sort_values("date")
        name = grp["site_name"].iloc[0] if "site_name" in grp.columns else site_no

        ax.scatter(grp["date"], grp["water_level_ft"],
                   color="#1A5276", s=8, alpha=0.5)

        # Annual median
        annual = grp.groupby("year")["water_level_ft"].median().reset_index()
        annual["date_mid"] = pd.to_datetime(annual["year"].astype(str) + "-07-01")
        ax.plot(annual["date_mid"], annual["water_level_ft"],
                color="#1A5276", linewidth=2, label="Annual median")

        # Trend line
        tr = gwl_trend_df[gwl_trend_df["site_no"] == site_no] if not gwl_trend_df.empty else pd.DataFrame()
        if not tr.empty:
            slope = tr["slope_ft_yr"].iloc[0]
            yrs   = annual["year"].values
            trend = slope * (yrs - yrs.mean()) + annual["water_level_ft"].mean()
            ax.plot(annual["date_mid"], trend, color="#C0392B",
                    linewidth=1.5, linestyle="--",
                    label=f"Trend: {tr['slope_ft_decade'].iloc[0]:+.2f} ft/decade")

        ax.invert_yaxis()  # depth increases downward
        ax.set_ylabel("Depth to water (ft below surface)", fontsize=8)
        ax.set_title(name[:50], fontsize=8, fontweight="bold")
        ax.legend(fontsize=7)
        sns.despine(ax=ax)

    for ax in axes[n:]:
        ax.set_visible(False)

    plt.suptitle(
        "USGS Groundwater Levels Near Pine Ridge and Rosebud\n"
        "Y-axis inverted: up = shallower water",
        fontsize=11, fontweight="bold",
    )
    plt.tight_layout()
    try:
        fig.savefig(fig_dir / "05_groundwater_level_timeseries.png",
                    dpi=150, bbox_inches="tight")
    except Exception:
        pass
    plt.show()
else:
    print("No water level time series to plot.")
    print("If this region has sparse USGS monitoring, that is itself a finding.")
    print("See the coverage gap map for the policy argument.")

## Exports

In [ ]:
try:
    constants.OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
except FileExistsError:
    pass

coverage_df.to_csv(
    constants.OUTPUTS_DIR/"groundwater_coverage_by_tribe.csv", index=False
)
print("Exported to outputs/groundwater_coverage_by_tribe.csv")

if not gwl_trend_df.empty:
    gwl_trend_df.to_csv(
        constants.OUTPUTS_DIR/"groundwater_level_trends.csv", index=False
    )
    print("Exported to outputs/groundwater_level_trends.csv")

if not sites_gdf.empty:
    sites_gdf[
        [c for c in ["site_no", "station_nm", "site_tp_cd",
                     "dec_lat_va", "dec_long_va", "begin_date", "end_date",
                     "geometry"]
         if c in sites_gdf.columns]
    ].to_file(
        constants.OUTPUTS_DIR/"nwis_gw_sites.geojson", driver="GeoJSON"
    )
    print("Exported to outputs/nwis_gw_sites.geojson")

## Summary and Findings

*(Fill in after running with your data.)*

**What the data shows:**
- How many USGS groundwater monitoring wells exist within each SD Tribal land?
  How many within 50 km? Which Tribal Nations fall in monitoring dead zones?
- For the wells with long-term records, is groundwater depth to water
  increasing (water table falling) or stable?
- Do the low water level years correspond to the major drought years
  identified in notebook 02 (PDSI)? That connection confirms the physical
  mechanism: drought to reduced recharge to deeper water table.

**The monitoring gap as equity finding:**
If reservation counties show fewer USGS monitoring wells per km² than
surrounding counties, document this explicitly. It means:
1. Federal investment in water monitoring infrastructure has been lower on
   Tribal lands than on surrounding lands.
2. The absence of data is not evidence of absence of groundwater stress.
3. Tribal-collected well data (operational pipeline) is essential because
   federal monitoring has not filled this gap.

**For Tribal decision makers:**
The coverage gap analysis supports arguments for:
- BIA Water Resources Program funding for Tribal monitoring wells
- EPA Clean Water Act Section 319 investments
- Tribal climate resilience plan water infrastructure components
- The necessity of the operational data collection pipeline in this repository

**Connection to the rest of the series:**
- Notebook 06 (System Stress Index) integrates groundwater trends with
  drought and NDVI to create a compound stress indicator
- The operational pipeline `pipeline/groundwater.py` processes Tribal-
  collected well level observations to fill the USGS gap at the field level

In [ ]:
# Print citations
print(generate_citations(["census_aiannh", "usgs_nwis_groundwater"]))